# RAG Pipeline Run

Build the RAG index and run a sample query.

Steps:
- Build the RAG index.
- Run a sample query for validation.
- Inspect embeddings output.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
summary = {
    'ingest_exit': None,
    'query_result': None,
    'embeddings': [],
}

run([PY, 'scripts/rag/build_index.py'])


In [ ]:
# Run a sample RAG query.
try:
    from app.rag.dsa_pipeline import answer_query
except Exception as exc:
    answer_query = None
    print('Could not import RAG pipeline:', exc)

if answer_query:
    result = answer_query('Summarize the main capabilities of this system.')
    summary['query_result'] = {
        'answer_preview': (result.get('answer') or '')[:400],
        'sources': len(result.get('sources', [])),
    }
    print(summary['query_result'])


In [ ]:
# Inspect embeddings output.
embeddings_dir = REPO_ROOT / 'data' / 'dsa_embeddings'
if embeddings_dir.exists():
    files = [p for p in embeddings_dir.rglob('*') if p.is_file()]
    summary['embeddings'] = [str(p.relative_to(REPO_ROOT)) for p in files[:10]]
    for item in summary['embeddings']:
        print(' -', item)
else:
    print('Missing:', embeddings_dir)


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'execution_rag_pipeline_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
